# Volatility Forecasting and Volatility Control: Walk-Forward Evaluation Results

This notebook runs the repository’s walk-forward volatility forecasting experiment and reports:

- **Forecast evaluation**: QLIKE on the variance scale, with RMSE and Spearman correlation on the volatility scale.
- **Diebold–Mariano (DM) tests vs baseline** (Newey-West HAC).
- **Daily-reset vol-control results under transaction costs** (no leverage; risky weight capped at 1).

The notebook is intentionally thin: all compute logic lives in `src/vol_forecast/`. This file is a reproducible driver that loads data, runs the experiment, and displays the resulting tables.


In [1]:
import numpy as np
import pandas as pd

from vol_forecast.config import ExperimentSpec
from vol_forecast.experiment import build_experiment_df, compute_experiment_report

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4g}")

def _fmt_cell(x):
    if isinstance(x, (np.floating, float)):
        x = float(x)
        return f"{x:,.0f}" if x.is_integer() else f"{x:,.4g}"
    return str(x)

def show_df(df: pd.DataFrame, formats=None) -> None:
    formatter = _fmt_cell if formats is None else formats
    display(df.style.hide(axis="index").format(formatter, na_rep="—"))


## Overview

### Target
We forecast **one-day variance**, using the next squared close-to-close return as a noisy realized-variance proxy and annualizing it using 252 trading days per year.

### Common sample

All models are compared from the first origin at which HAR has 1,000 eligible training observations after its 22-day feature warmup. The other forecasts are available earlier but are restricted to the same dates.

### Baseline and models
**Baseline:** `ewma_forecast` (RiskMetrics EWMA applied to lagged daily squared returns with the fixed decay factor $\lambda=0.94$).

**Models compared:**
- EWMA
- HAR (1/5/22-day range-variance components)
- GARCH and GJR-GARCH (Student-t innovations)

Conventional HAR-RV models forecast daily realized variance, usually estimated as the sum of squared intraday returns. Here, the HAR predictors are constructed from a daily range-based variance proxy combining the `^GSPC` squared overnight return and the Parkinson high-low estimator:

$$
RV^{range}_t = 252\left[\log^2\left(O_t/C_{t-1}\right) + \frac{\log^2(H_t/L_t)}{4\log 2}\right].
$$

Its lagged 1/5/22-day components are used to forecast the common next day squared return target.

Also, we retain GARCH family estimates that converge at the unit persistence boundary even though they do not imply a finite unconditional variance because the conditional variance forecasts used here remain well defined. We only reject invalid estimates or estimates with persistence above one.

### Forecast evaluation metrics
- **Primary — QLIKE on the variance scale.** For realized variance $y_t$ and forecast $h_t$,

$$
QLIKE(y_t,h_t)=\frac{y_t}{h_t}-\log\left(\frac{y_t}{h_t}\right)-1.
$$

QLIKE is a proper scoring rule for conditional variance forecasts: its expected value is minimized by the true conditional variance. It is sensitive to forecast level and asymmetric (penalizes underforecasting more than overforecasting). We treat the conditional mean of daily returns as negligible, so

$$
E[r_t^2\mid\mathcal F_{t-1}] \approx \operatorname{Var}(r_t\mid\mathcal F_{t-1}),
$$

making squared returns a noisy proxy for conditional variance.

- **Secondary — RMSE on the volatility scale.** This summarizes forecast magnitude errors after converting variance to volatility.
- **Secondary — Spearman correlation on the volatility scale.** This measures how well forecasts rank lower and higher volatility states.

QLIKE determines the headline forecast ordering. RMSE and Spearman help identify whether differences arise from volatility magnitude errors or regime ordering.

The DM tests below compare each model's average QLIKE with EWMA, using Newey-West standard errors at lags 0, 5, and 20 to allow for serial dependence in the loss differential.

### Strategy layer (daily-reset capped vol-control; no leverage)

The vol-control backtest allocates between the risky equity index and a cash proxy: risky weight $w_t$, remainder held in cash $1-w_t$.

We map forecast volatility $(\hat\sigma_t = \sqrt{\widehat{\mathrm{var}}_t})$ into the risky weight via:

$$
w_t = \min\left(1,\ \sigma_{\text{target}}/\hat\sigma_t\right).
$$

The target weight is reset daily, aligning the decision frequency with the one-day forecast horizon. The leverage cap, forecast errors and realized return path can still cause achieved volatility to differ from $\sigma_{\text{target}}$.




## Experiment configuration

Key settings are defined in the next cell:

- **Data window:** `data_start`, `data_end`
- **Walk-forward estimation:** `rolling_window`, `refit_every`
- **DM sensitivity:** `hac_lags`
- **Strategy:** `target_volatility`, `costs_bps`


In [2]:
spec = ExperimentSpec(
    data_start="2004-01-01",
    data_end="2026-02-06",
    rolling_window=1000,
    refit_every=60,
    hac_lags=(0, 5, 20),
    target_volatility=0.10,
    costs_bps=(0.0, 1.0, 5.0),
)


## Build canonical experiment DataFrame

We build the canonical experiment dataset:

- `base_df`: aligned returns, cash proxy, S&P 500 OHLC data, one-day variance target, lagged HAR components, and baseline forecasts.



In [3]:
base_df = build_experiment_df(spec)
base_df.shape, base_df.index.min(), base_df.index.max()


((5558, 13),
 Timestamp('2004-01-05 00:00:00'),
 Timestamp('2026-02-05 00:00:00'))

## Forecast results

This section reports out-of-sample forecasting performance over the **common walk-forward sample**.

In [4]:
report = compute_experiment_report(base_df, spec)


### Forecast metrics

The table is ordered by QLIKE, the primary forecast metric.

Note:
- `delta_qlike_vs_ewma < 0` means the model improves on the EWMA baseline over the same evaluation dates.

In [5]:
show_df(report["forecast_metrics"])


model,n,qlike,delta_qlike_vs_ewma,rmse_vol,spearman_vol
gjr_forecast,4535,1.573,-0.09261,0.1368,0.4245
har_forecast,4535,1.585,-0.08011,0.1496,0.4111
garch_forecast,4535,1.618,-0.04715,0.1389,0.3784
ewma_forecast,4535,1.665,0,0.1416,0.3615


### What the forecast metrics show in this run

- GJR has the lowest QLIKE, vol RMSE and also the highest Spearman correlation.
- HAR ranks second by both QLIKE and Spearman but has the highest vol RMSE.
- GARCH ranks second on vol RMSE and does better than the baseline on all metrics.


## QLIKE loss comparisons

The table below compares each model's QLIKE loss differential with the EWMA baseline.

### DM tests vs baseline (HAC)

We report Diebold–Mariano (DM) tests comparing each model’s QLIKE loss series to EWMA over the common walk-forward sample.

- We define the loss differential at time $t$ as  
  $d_t = \ell_{model,t} - \ell_{baseline,t}$, where $\ell$ is QLIKE on the variance scale.  
  Negative $\overline{d}$ implies the model has lower average QLIKE than the baseline.
- The one-day targets do not overlap, but volatility forecasts and their loss differentials can remain serially dependent. We therefore show lag 0 together with 5- and 20-day Newey–West estimates as weekly and approximately monthly sensitivity checks.
- **Interpretation:** `mean_d` conveys direction + effect size; `dm_stat` / `p_value` summarize evidence relative to HAC-estimated noise.  
  DM is used to describe uncertainty around the QLIKE loss differentials, not as a model-selection criterion. P-values are **unadjusted** and shown for context only. Given multiple model-vs-baseline comparisons, they shouldn't be read against a hard 5% threshold; strict inference would require a multiple-testing correction such as **Holm**.

In [6]:
show_df(report["dm"])


model,hac_lag,n,mean_d,dm_stat,p_value
garch_forecast,0,4535,-0.04715,-3.9,9.61e-05
garch_forecast,5,4535,-0.04715,-3.616,0.0002995
garch_forecast,20,4535,-0.04715,-3.504,0.0004588
gjr_forecast,0,4535,-0.09261,-4.522,6.117e-06
gjr_forecast,5,4535,-0.09261,-4.204,2.617e-05
gjr_forecast,20,4535,-0.09261,-4.037,5.404e-05
har_forecast,0,4535,-0.08011,-4.53,5.912e-06
har_forecast,5,4535,-0.08011,-4.294,1.756e-05
har_forecast,20,4535,-0.08011,-4.197,2.707e-05


### What the DM tests show in this run

- GARCH, GJR, and HAR have negative average loss differentials at every reported lag.
- These directions and the reported p-value conclusions are unchanged across the HAC grid.


## Strategy evaluation: daily-reset vol-control under transaction costs


We apply the same capped daily-reset vol-control policy to each volatility forecast over the common walk-forward sample.

`excess_sharpe` uses daily returns above the lagged DFF cash proxy, while `delta_sharpe_vs_ewma` compares results under the same transaction-cost assumption. `annual_turnover` is one-way turnover in portfolio-value multiples per year, and `pct_capped` is the share of daily target weights capped at 100% equity.

### Results

We show buy-and-hold once, followed by results at `cost_bps` in {0, 1, 5}, sorted by excess Sharpe within each cost level. Costs are one-way basis points per traded notional.


In [7]:
strategy = report["strategy"]
strategy_formats = {
    "cost_bps": "{:.0f}",
    "annual_return": "{:.2%}",
    "excess_sharpe": "{:.3f}",
    "delta_sharpe_vs_ewma": "{:.3f}",
    "realized_volatility": "{:.2%}",
    "max_drawdown": "{:.2%}",
    "average_equity_weight": "{:.2%}",
    "annual_turnover": "{:.2f}x",
    "pct_capped": "{:.2%}",
}
show_df(
    strategy[strategy["model"] == "buy_and_hold"],
    formats=strategy_formats,
)

for cost in spec.costs_bps:
    show_df(
        strategy[
            strategy["cost_bps"].eq(cost)
            & strategy["model"].ne("buy_and_hold")
        ]
        .sort_values("excess_sharpe", ascending=False)
        .reset_index(drop=True),
        formats=strategy_formats,
    )


cost_bps,model,annual_return,excess_sharpe,delta_sharpe_vs_ewma,realized_volatility,max_drawdown,average_equity_weight,annual_turnover,pct_capped
—,buy_and_hold,11.51%,0.575,—,19.99%,-51.52%,100.00%,0.00x,—


cost_bps,model,annual_return,excess_sharpe,delta_sharpe_vs_ewma,realized_volatility,max_drawdown,average_equity_weight,annual_turnover,pct_capped
0,garch_forecast,8.44%,0.740,0.033,9.69%,-18.98%,69.23%,8.29x,14.73%
0,har_forecast,8.16%,0.732,0.025,9.40%,-20.36%,68.44%,14.28x,14.82%
0,ewma_forecast,8.32%,0.707,0.000,10.03%,-18.49%,70.93%,4.24x,19.45%
0,gjr_forecast,8.06%,0.706,-0.001,9.65%,-19.42%,71.02%,9.24x,22.58%


cost_bps,model,annual_return,excess_sharpe,delta_sharpe_vs_ewma,realized_volatility,max_drawdown,average_equity_weight,annual_turnover,pct_capped
1,garch_forecast,8.35%,0.731,0.028,9.70%,-19.01%,69.23%,8.29x,14.73%
1,har_forecast,8.00%,0.717,0.014,9.40%,-20.39%,68.44%,14.28x,14.82%
1,ewma_forecast,8.27%,0.703,0.000,10.03%,-18.51%,70.93%,4.24x,19.45%
1,gjr_forecast,7.96%,0.697,-0.006,9.65%,-19.45%,71.02%,9.24x,22.58%


cost_bps,model,annual_return,excess_sharpe,delta_sharpe_vs_ewma,realized_volatility,max_drawdown,average_equity_weight,annual_turnover,pct_capped
5,garch_forecast,7.99%,0.697,0.011,9.70%,-19.09%,69.23%,8.29x,14.73%
5,ewma_forecast,8.09%,0.686,0.000,10.03%,-18.58%,70.93%,4.24x,19.45%
5,gjr_forecast,7.56%,0.658,-0.028,9.65%,-19.55%,71.02%,9.24x,22.58%
5,har_forecast,7.39%,0.656,-0.030,9.40%,-20.52%,68.44%,14.28x,14.82%


### What the strategy tables show in this run

- All vol control strategies reduce volatility from around 20% for the simple buy and hold to around the 10% risk budget. GARCH, GJR and HAR finish below it whereas EWMA finishes approximately at it.

- GJR produces lower realized volatility than EWMA with nearly the same average exposure which hints at better risk timing, whereas GARCH and HAR use less. EWMA finishes approximately at the 10% risk budget despite the no leverage constraint, which likely reflects a tendency to underpredict volatility.

- GARCH has the highest excess sharpe at every cost. HAR has highest turnover and falls from second place with 0 bps costs to last at 5 bps. EWMA's lower turnover makes it more competitive as costs rise.

## Conclusion

All four strategies reduce realized volatility to around or below the 10% risk budget. GJR has the lowest QLIKE and delivers lower realized volatility than EWMA at nearly the same average exposure. GARCH has the highest excess Sharpe at every cost level while remaining below budget whereas EWMA has the lowest turnover and drawdowns.

The main result is that more accurate variance forecasts do not necessarily produce better volatility-control outcomes. Greater accuracy helps avoid exceeding the 10% volatility ceiling, but it does not consistently translate into better risk budget use, higher Sharpe, lower turnover or smaller drawdowns.